### What it does
Script to handle words with "-", for example "risk-taker". The aim is to modify it into "risk taker", "risktaker". 
It searchs "-" in words ignoring case. Then it split the words by "-" and joint the words in one taking into account lowcase letter.
It creates new table with the the same columns as original table. Original table with new one are merged and sorted by lemma. 

In [5]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

# pandas library to work with tables/dataframes
import pandas as pd

# Regular expression library
import re
import ast

In [6]:
#file name
file_name = "rules_16FEB_German"

In [7]:
df_rules = pd.read_csv(file_name+".csv")

In [8]:
#check first 5 rows
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,...,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives,Sample_Sentences,False_Positives,Rule copy,3rd_party_source,ToClarify,Notes
0,jammern,v,5/1/2022,gendered,female_stereotype,culture,NaN,kritisieren,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Lehrtochter,s,NaN,gendered,female_stereotype,NaN,Ausbildung | Lehre,Nachwuchskraft | Azubi,Auszubildend~e~/~Auszubildender,Person in Ausbildung | auszubildende Person | ...,...,Lernende | Auszubildende,NaN,Personen in Ausbildung | auszubildende Persone...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,zickig,a,NaN,gendered,female_stereotype,NaN,NaN,anderer Ansicht | uneinsichtig,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Alpha,s,NaN,style,exaggerating,NaN,NaN,NaN,NaN,"Person, die ihr Team inspiriert | Mensch mit F...",...,NaN,NaN,"Leute, die ihre Teams inspirieren | Menschen m...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Alpha-Chef,s,NaN,style,exaggerating,NaN,NaN,NaN,NaN,"Person, die ihr Team inspiriert | Mensch mit F...",...,NaN,NaN,"Leute, die ihre Teams inspirieren | Menschen m...",NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Remove unnecessary columns

In [9]:
#create a subset of dataframe from the 'Lemma' till the last column with alternatives '3rd_party_alternatives'
df_rules=df_rules.loc[:, 'Lemma':'3rd_party_alternatives']

In [10]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1700 entries, 0 to 1699
Data columns (total 16 columns):
Lemma                             1700 non-null object
Word_Type                         1699 non-null object
Priority                          213 non-null object
Category                          1698 non-null object
Primary_subcategory               1698 non-null object
Secondary_subcategory             452 non-null object
Alt_Field                         785 non-null object
Alt_Sg_Replacement                539 non-null object
Alt_Sg_/_and_inclusive_form       269 non-null object
Alt_Sg_people_first               172 non-null object
Alt_Sg_reframed                   191 non-null object
Alt_Pl_collective_noun            45 non-null object
Alt_Pl                            146 non-null object
Alt_Pl_pair_and_inclusive_form    235 non-null object
Alt_Pl_people_first               108 non-null object
3rd_party_alternatives            1 non-null object
dtypes: object(16)
memory us

In [11]:
#check the number of columns
len(list(df_rules.columns))

16

In [34]:
# delete rows with empty values in "Lemma", "Word_Type", 'Category', "Primary_subcategory"  
df_rules = df_rules.dropna(subset = ["Lemma", "Word_Type", 'Category', "Primary_subcategory"])

In [35]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1698 entries, 0 to 1699
Data columns (total 16 columns):
Lemma                             1698 non-null object
Word_Type                         1698 non-null object
Priority                          212 non-null object
Category                          1698 non-null object
Primary_subcategory               1698 non-null object
Secondary_subcategory             452 non-null object
Alt_Field                         785 non-null object
Alt_Sg_Replacement                538 non-null object
Alt_Sg_/_and_inclusive_form       269 non-null object
Alt_Sg_people_first               172 non-null object
Alt_Sg_reframed                   191 non-null object
Alt_Pl_collective_noun            45 non-null object
Alt_Pl                            146 non-null object
Alt_Pl_pair_and_inclusive_form    235 non-null object
Alt_Pl_people_first               108 non-null object
3rd_party_alternatives            1 non-null object
dtypes: object(16)
memory us

### Splitting and merging words by hyphen

In [14]:
list_word = []

list_word_type = []
list_priority = []
list_category = []
list_primary_subcategory = []
list_secondary_subcategory = []
list_alt_field= []
list_alt_sg_replacement = []
list_alt_sg_and_inclusive_form = []
list_alt_sg_people_first = []
list_alt_sg_reframed = []
list_alt_pl_collective_noun = []
list_alt_pl = []
list_alt_pl_pair_and_inclusive_form = []
list_alt_pl_people_first = []
list_notes = []

list_3rd_party_alternatives = []
list_3rd_party_source = []
list_toclarify = []

for index, row in df_rules.iterrows():
    word = row["Lemma"]
    span = re.search("-", word)
    #filter words with only one "-"
    if type(span) == re.Match and len(word.rsplit("-"))==2:
        #merge the words by "-" and save the results in lists
        list_word.append(word.rsplit("-")[0]+word.rsplit("-")[1].lower())
        
        list_word_type.append(row["Word_Type"])
        list_priority.append(row["Priority"])
        list_category.append(row["Category"])
        list_primary_subcategory.append(row["Primary_subcategory"])
        list_secondary_subcategory.append(row["Secondary_subcategory"])
        list_alt_field.append(row["Alt_Field"])
        list_alt_sg_replacement.append(row["Alt_Sg_Replacement"])
        list_alt_sg_and_inclusive_form.append(row["Alt_Sg_/_and_inclusive_form"])
        list_alt_sg_people_first.append(row["Alt_Sg_people_first"])
        list_alt_sg_reframed.append(row["Alt_Sg_reframed"])
        list_alt_pl_collective_noun.append(row["Alt_Pl_collective_noun"])
        list_alt_pl.append(row["Alt_Pl"])
        list_alt_pl_pair_and_inclusive_form.append(row["Alt_Pl_pair_and_inclusive_form"])
        list_alt_pl_people_first.append(row["Alt_Pl_people_first"])
        list_3rd_party_alternatives.append(row["3rd_party_alternatives"])
        
        
        #split the words by "-", and then save it to the list
        list_word.append(word.rsplit("-")[0]+" "+word.rsplit("-")[1])
        list_word_type.append(row["Word_Type"])
        list_priority.append(row["Priority"])
        list_category.append(row["Category"])
        list_primary_subcategory.append(row["Primary_subcategory"])
        list_secondary_subcategory.append(row["Secondary_subcategory"])
        list_alt_field.append(row["Alt_Field"])
        list_alt_sg_replacement.append(row["Alt_Sg_Replacement"])
        list_alt_sg_and_inclusive_form.append(row["Alt_Sg_/_and_inclusive_form"])
        list_alt_sg_people_first.append(row["Alt_Sg_people_first"])
        list_alt_sg_reframed.append(row["Alt_Sg_reframed"])
        list_alt_pl_collective_noun.append(row["Alt_Pl_collective_noun"])
        list_alt_pl.append(row["Alt_Pl"])
        list_alt_pl_pair_and_inclusive_form.append(row["Alt_Pl_pair_and_inclusive_form"])
        list_alt_pl_people_first.append(row["Alt_Pl_people_first"])  
        list_3rd_party_alternatives.append(row["3rd_party_alternatives"])
        
        

### New created dataframe

In [23]:
data = pd.DataFrame(list(zip(list_word,list_word_type,list_priority, list_category, list_primary_subcategory, list_secondary_subcategory, 
list_alt_field, list_alt_sg_replacement, list_alt_sg_and_inclusive_form, list_alt_sg_people_first, list_alt_sg_reframed, list_alt_pl_collective_noun,
list_alt_pl, list_alt_pl_pair_and_inclusive_form, list_alt_pl_people_first, list_3rd_party_alternatives)), columns=list(df_rules.columns))

In [16]:
# check first 5 rows of new dataframe
data.head(5)

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives
0,Alphachef,s,NaN,style,exaggerating,NaN,NaN,NaN,NaN,"Person, die ihr Team inspiriert | Mensch mit F...",NaN,NaN,NaN,NaN,"Leute, die ihre Teams inspirieren | Menschen m...",NaN
1,Alpha Chef,s,NaN,style,exaggerating,NaN,NaN,NaN,NaN,"Person, die ihr Team inspiriert | Mensch mit F...",NaN,NaN,NaN,NaN,"Leute, die ihre Teams inspirieren | Menschen m...",NaN
2,Alphaführungskraft,s,NaN,style,exaggerating,NaN,NaN,NaN,NaN,"Person, die ihr Team inspiriert | Mensch mit F...",NaN,NaN,NaN,NaN,"Leute, die ihre Teams inspirieren | Menschen m...",NaN
3,Alpha Führungskraft,s,NaN,style,exaggerating,NaN,NaN,NaN,NaN,"Person, die ihr Team inspiriert | Mensch mit F...",NaN,NaN,NaN,NaN,"Leute, die ihre Teams inspirieren | Menschen m...",NaN
4,Alphaperson,s,NaN,style,exaggerating,NaN,NaN,NaN,NaN,"Person, die ihr Team inspiriert | Mensch mit F...",NaN,NaN,NaN,NaN,"Leute, die ihre Teams inspirieren | Menschen m...",NaN


In [17]:
#general info about new dataframe
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 16 columns):
Lemma                             94 non-null object
Word_Type                         92 non-null object
Priority                          50 non-null object
Category                          92 non-null object
Primary_subcategory               92 non-null object
Secondary_subcategory             12 non-null object
Alt_Field                         14 non-null object
Alt_Sg_Replacement                42 non-null object
Alt_Sg_/_and_inclusive_form       12 non-null object
Alt_Sg_people_first               28 non-null object
Alt_Sg_reframed                   2 non-null object
Alt_Pl_collective_noun            2 non-null object
Alt_Pl                            8 non-null object
Alt_Pl_pair_and_inclusive_form    10 non-null object
Alt_Pl_people_first               24 non-null object
3rd_party_alternatives            2 non-null object
dtypes: object(16)
memory usage: 11.9+ KB


### Add new dataframe to existing one

In [18]:
df_fin = df_rules.append(data, ignore_index = True)

In [44]:
#check final dataframe
df_fin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1790 entries, 0 to 1789
Data columns (total 16 columns):
Lemma                             1790 non-null object
Word_Type                         1790 non-null object
Priority                          260 non-null object
Category                          1790 non-null object
Primary_subcategory               1790 non-null object
Secondary_subcategory             464 non-null object
Alt_Field                         799 non-null object
Alt_Sg_Replacement                580 non-null object
Alt_Sg_/_and_inclusive_form       281 non-null object
Alt_Sg_people_first               200 non-null object
Alt_Sg_reframed                   193 non-null object
Alt_Pl_collective_noun            47 non-null object
Alt_Pl                            154 non-null object
Alt_Pl_pair_and_inclusive_form    245 non-null object
Alt_Pl_people_first               132 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us

In [45]:
#check last 5 rows of new dataframe since the new data were added by the end
df_fin.tail()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives
1785,Trans Frau,s,12/2/2022,unconscious_bias,gender_identity,NaN,NaN,trans Frau,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1786,Transmann,s,12/2/2022,unconscious_bias,gender_identity,NaN,NaN,trans Mann,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1787,Trans Mann,s,12/2/2022,unconscious_bias,gender_identity,NaN,NaN,trans Mann,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1788,Homoehe,s,12/2/2022,unconscious_bias,sexual_orientation,NaN,NaN,Ehe für alle,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1789,Homo Ehe,s,12/2/2022,unconscious_bias,sexual_orientation,NaN,NaN,Ehe für alle,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Sorted alphabeticaly "Lemma"

In [19]:
df_sorted = df_fin.sort_values(by=['Lemma'], inplace=False, ascending=True)

In [20]:
#check sorted dataframe
df_sorted.head(5)

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives
137,Abdichter,s,NaN,gendered,titles,NaN,Abdichtung,NaN,Abdichter~in~/~Abdichter,NaN,NaN,NaN,NaN,Abdichter~innen~ und ~Abdichter,NaN,NaN
1569,Abgeordneter,s,26-01-2022,gendered,titles,NaN,NaN,NaN,Abgeordnet~e~/~Abgeordneter,NaN,NaN,NaN,Abgeordnete,NaN,NaN,NaN
303,Abhängigkeit,s,NaN,inclusive,communal,caring,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
653,Abwichshure,s,NaN,openly_discriminating,sexism,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
304,Achtsamkeit,s,NaN,inclusive,communal,diligence,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Save to csv file after preprocessing

In [21]:
df_sorted.to_csv(file_name+"_h"+".csv", index=False)

In [22]:
df_test = pd.read_csv(file_name+"_h"+".csv")

In [51]:
df_test.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives
0,Abdichter,s,NaN,gendered,titles,NaN,Abdichtung,NaN,Abdichter~in~/~Abdichter,NaN,NaN,NaN,NaN,Abdichter~innen~ und ~Abdichter,NaN,NaN
1,Abgeordneter,s,26-01-2022,gendered,titles,NaN,NaN,NaN,Abgeordnet~e~/~Abgeordneter,NaN,NaN,NaN,Abgeordnete,NaN,NaN,NaN
2,Abhängigkeit,s,NaN,inclusive,communal,caring,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Abwichshure,s,NaN,openly_discriminating,sexism,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Achtsamkeit,s,NaN,inclusive,communal,diligence,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1791 entries, 0 to 1790
Data columns (total 16 columns):
Lemma                             1790 non-null object
Word_Type                         1790 non-null object
Priority                          260 non-null object
Category                          1790 non-null object
Primary_subcategory               1790 non-null object
Secondary_subcategory             464 non-null object
Alt_Field                         799 non-null object
Alt_Sg_Replacement                580 non-null object
Alt_Sg_/_and_inclusive_form       281 non-null object
Alt_Sg_people_first               200 non-null object
Alt_Sg_reframed                   193 non-null object
Alt_Pl_collective_noun            47 non-null object
Alt_Pl                            154 non-null object
Alt_Pl_pair_and_inclusive_form    245 non-null object
Alt_Pl_people_first               132 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us